In [0]:
# SETUP

 
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import date, timedelta
 
spark.sql("CREATE DATABASE IF NOT EXISTS tijartek_gold")
print(" tijartek_gold database ready")
print(" Imports done")

 tijartek_gold database ready
 Imports done


In [0]:
# DIM_DATE
# Generated from scratch - full date spine 2020 to 2027
 
start = date(2020, 1, 1)
end   = date(2027, 12, 31)
dates = [(start + timedelta(days=i),) for i in range((end - start).days + 1)]
 
dim_date = spark.createDataFrame(dates, ["full_date"]) \
    .withColumn("full_date",      F.col("full_date").cast(DateType())) \
    .withColumn("date_key",       F.date_format("full_date", "yyyyMMdd").cast(IntegerType())) \
    .withColumn("day",            F.dayofmonth("full_date").cast(ByteType())) \
    .withColumn("month",          F.month("full_date").cast(ByteType())) \
    .withColumn("quarter",        F.quarter("full_date").cast(ByteType())) \
    .withColumn("year",           F.year("full_date").cast(ShortType())) \
    .withColumn("week_number",    F.weekofyear("full_date").cast(ByteType())) \
    .withColumn("day_name",       F.date_format("full_date", "EEEE")) \
    .withColumn("month_name",     F.date_format("full_date", "MMMM")) \
    .withColumn("is_weekend",     F.when(F.dayofweek("full_date").isin(1, 7), F.lit(1)).otherwise(F.lit(0)).cast(ByteType())) \
    .withColumn("is_holiday",     F.lit(0).cast(ByteType())) \
    .withColumn("fiscal_quarter", F.concat(F.lit("FQ"), F.quarter("full_date").cast(StringType()))) \
    .select(
        "date_key", "full_date", "day", "month", "quarter", "year",
        "week_number", "day_name", "month_name", "is_weekend",
        "is_holiday", "fiscal_quarter"
    )
 
dim_date.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_date")
 
print(f" dim_date: {dim_date.count():,} rows")

 dim_date: 2,922 rows


In [0]:
# DIM_LOCATION
# Columns: location_key, location_id, city, region, country, timezone
 
dim_location = spark.table("tijartek_silver.location") \
    .withColumn("location_key", F.col("location_id").cast(IntegerType())) \
    .withColumn("location_id",  F.col("location_id").cast(StringType())) \
    .select(
        "location_key", "location_id", "city", "region", "country", "timezone"
    )
 
dim_location.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_location")
 
print(f" dim_location: {dim_location.count():,} rows")
 

 dim_location: 50 rows


In [0]:
# DIM_PRODUCT (with category embedded)
# Columns: product_key, product_id, seller_key, category_key, name,
#          description, price, weight, is_fragile, category_name,
#          parent_category, commission_rate, is_current, effective_from, effective_to

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, ByteType, DateType

dim_product = spark.table("tijartek_silver.product") \
    .withColumn("product_key",   F.col("product_id").cast(IntegerType())) \
    .withColumn("product_id",    F.col("product_id").cast(StringType())) \
    .withColumn("seller_key",    F.col("seller_id").cast(IntegerType())) \
    .withColumn("category_key",  F.col("category_id").cast(IntegerType())) \
    .join(
        spark.table("tijartek_silver.category").select(
            F.col("category_id").alias("cat_id"),
            F.col("name").alias("category_name"),
            F.col("commission_rate")
        ),
        F.col("category_id") == F.col("cat_id"),
        how="left"
    ) \
    .withColumn("parent_category", F.lit(None).cast(StringType())) \
    .withColumn("is_current",      F.lit(1).cast(ByteType())) \
    .withColumn("effective_from",  F.current_date()) \
    .withColumn("effective_to",    F.lit("9999-12-31").cast(DateType())) \
    .select(
        "product_key", "product_id", "seller_key", "category_key",
        "name", "description", "price", "weight", "is_fragile",
        "category_name", "parent_category", "commission_rate",
        "is_current", "effective_from", "effective_to"
    )

dim_product.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_product")

print(f" dim_product: {dim_product.count():,} rows")

 dim_product: 50,000 rows


In [0]:
# DIM_PROMOTION
# Columns: promotion_key, promotion_id, name, discount_type, platform, start_date, end_date, budget
 
dim_promotion = spark.table("tijartek_silver.promotions") \
    .withColumn("promotion_key", F.col("promotion_id").cast(IntegerType())) \
    .withColumn("promotion_id",  F.col("promotion_id").cast(StringType())) \
    .select(
        "promotion_key", "promotion_id", "name", "discount_type",
        "platform", "start_date", "end_date", "budget"
    )
 
dim_promotion.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_promotion")
 
print(f" dim_promotion: {dim_promotion.count():,} rows")

 dim_promotion: 500 rows


In [0]:
#  DIM_SELLER
# Columns: seller_key, seller_id, location_key, name, type, join_date, seller_tier
 
dim_seller = spark.table("tijartek_silver.seller") \
    .withColumn("seller_key",   F.col("seller_id").cast(IntegerType())) \
    .withColumn("seller_id",    F.col("seller_id").cast(StringType())) \
    .withColumn("location_key", F.col("location_id").cast(IntegerType())) \
    .select(
        "seller_key", "seller_id", "location_key",
        "name", "type", "join_date", "seller_tier"
    )
 
dim_seller.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_seller")
 
print(f" dim_seller: {dim_seller.count():,} rows")

 dim_seller: 1,000 rows


In [0]:
# DIM_CUSTOMER (SCD Type 2)
# Columns: customer_key, customer_id, location_key, name, email, phone, gender, age_band, registration_date, is_current, effective_from, effective_to
 
dim_customer = spark.table("tijartek_silver.customer") \
    .withColumn("customer_key",   F.col("customer_id").cast(IntegerType())) \
    .withColumn("customer_id",    F.col("customer_id").cast(StringType())) \
    .withColumn("location_key",   F.col("location_id").cast(IntegerType())) \
    .withColumn("is_current",     F.lit(1).cast(ByteType())) \
    .withColumn("effective_from", F.col("registration_date").cast(DateType())) \
    .withColumn("effective_to",   F.lit("9999-12-31").cast(DateType())) \
    .select(
        "customer_key", "customer_id", "location_key",
        "name", "email", "phone", "gender", "age_band",
        "registration_date", "is_current", "effective_from", "effective_to"
    )
 
dim_customer.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_customer")
 
print(f" dim_customer: {dim_customer.count():,} rows")

 dim_customer: 80,000 rows


In [0]:
#  DIM_PAYMENT (Bridge)
# Columns: payment_key, order_id, date_key, method, status, amount
 
payment  = spark.table("tijartek_silver.payment")
dim_date = spark.table("tijartek_gold.dim_date")
 
dim_payment = payment \
    .join(
        dim_date.select(F.col("date_key"), F.col("full_date").alias("pay_date")),
        F.to_date(F.col("payment_date")) == F.col("pay_date"),
        how="left"
    ) \
    .withColumn("payment_key", F.col("payment_id").cast(LongType())) \
    .withColumn("order_id",    F.col("order_id").cast(LongType())) \
    .select(
        "payment_key", "order_id", "date_key", "method", "status", "amount"
    )
 
dim_payment.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_payment")
 
print(f" dim_payment (bridge): {dim_payment.count():,} rows")
 

 dim_payment (bridge): 150,000 rows


In [0]:
# DIM_REVIEW (Bridge)
# Columns: review_key, product_key, customer_key, date_key, rating, sentiment
 
review   = spark.table("tijartek_silver.review")
dim_date = spark.table("tijartek_gold.dim_date")
 
dim_review = review \
    .join(
        dim_date.select(F.col("date_key"), F.col("full_date").alias("rev_date")),
        F.col("date") == F.col("rev_date"),
        how="left"
    ) \
    .withColumn("review_key",   F.col("review_id").cast(LongType())) \
    .withColumn("product_key",  F.col("product_id").cast(IntegerType())) \
    .withColumn("customer_key", F.col("customer_id").cast(IntegerType())) \
    .select(
        "review_key", "product_key", "customer_key",
        "date_key", "rating", "sentiment"
    )
 
dim_review.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.dim_review")
 
print(f" dim_review (bridge): {dim_review.count():,} rows")

 dim_review (bridge): 60,000 rows


In [0]:
# FACT_SALES
# Columns: sales_key, date_key, customer_key, product_key, seller_key,
#          location_key, promotion_key, order_id, quantity_sold,
#          unit_price, tax_amount, discount_amount, gross_revenue,
#          net_revenue, shipping_fee
 
order_item = spark.table("tijartek_silver.order_item")
order      = spark.table("tijartek_silver.order")
product    = spark.table("tijartek_silver.product")
shipment   = spark.table("tijartek_silver.shipment")
dim_date   = spark.table("tijartek_gold.dim_date")
 
shipment_location = shipment \
    .select("order_id", F.col("location_id").alias("delivery_location_id")) \
    .dropDuplicates(["order_id"])
 
fact_sales = order_item \
    .join(
        order.select(
            "order_id", "customer_id", "promotion_id",
            "order_date", "shipping_fee", "discount"
        ),
        on="order_id", how="left"
    ) \
    .join(
        product.select("product_id", "seller_id"),
        on="product_id", how="left"
    ) \
    .join(shipment_location, on="order_id", how="left") \
    .join(
        dim_date.select(F.col("date_key"), F.col("full_date")),
        F.col("order_date") == F.col("full_date"),
        how="left"
    ) \
    .withColumn("sales_key",       F.col("order_item_id").cast(LongType())) \
    .withColumn("customer_key",    F.col("customer_id").cast(IntegerType())) \
    .withColumn("product_key",     F.col("product_id").cast(IntegerType())) \
    .withColumn("seller_key",      F.col("seller_id").cast(IntegerType())) \
    .withColumn("location_key",    F.col("delivery_location_id").cast(IntegerType())) \
    .withColumn("promotion_key",   F.col("promotion_id").cast(IntegerType())) \
    .withColumn("quantity_sold",   F.col("quantity").cast(IntegerType())) \
    .withColumn("discount_amount", F.col("discount").cast(DecimalType(10, 2))) \
    .withColumn("net_revenue",     F.round(F.col("gross_revenue") - F.col("tax_amount"), 2)) \
    .select(
        "sales_key", "date_key", "customer_key", "product_key",
        "seller_key", "location_key", "promotion_key",
        F.col("order_id").cast(LongType()).alias("order_id"),
        "quantity_sold", "unit_price", "tax_amount",
        "discount_amount", "gross_revenue", "net_revenue", "shipping_fee"
    )
 
fact_sales.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.fact_sales")
 
print(f" fact_sales: {fact_sales.count():,} rows")

 fact_sales: 338,051 rows


In [0]:
# FACT_SHIPMENTS
# Columns: shipment_key, order_id, shipped_date_key, delivered_date_key,
#          location_key, status, delivery_days, is_on_time
 
shipment = spark.table("tijartek_silver.shipment")
dim_date = spark.table("tijartek_gold.dim_date")
 
fact_shipments = shipment \
    .join(
        dim_date.select(
            F.col("date_key").alias("shipped_date_key"),
            F.col("full_date").alias("sd")
        ),
        F.col("shipped_date") == F.col("sd"), how="left"
    ) \
    .join(
        dim_date.select(
            F.col("date_key").alias("delivered_date_key"),
            F.col("full_date").alias("dd")
        ),
        F.col("delivered_date") == F.col("dd"), how="left"
    ) \
    .withColumn("shipment_key", F.col("shipment_id").cast(LongType())) \
    .withColumn("location_key", F.col("location_id").cast(IntegerType())) \
    .select(
        "shipment_key",
        F.col("order_id").cast(LongType()).alias("order_id"),
        "shipped_date_key", "delivered_date_key",
        "location_key", "status", "delivery_days", "is_on_time"
    )
 
fact_shipments.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.fact_shipments")
 
print(f" fact_shipments: {fact_shipments.count():,} rows")

 fact_shipments: 131,981 rows


In [0]:
# FACT_RETURNS
# Columns: return_key, order_item_id, product_key, customer_key,
#          return_date_key, status, refund_amount
 
returns    = spark.table("tijartek_silver.return")
order_item = spark.table("tijartek_silver.order_item")
order      = spark.table("tijartek_silver.order")
dim_date   = spark.table("tijartek_gold.dim_date")
 
fact_returns = returns \
    .join(
        order_item.select("order_item_id", "product_id", "order_id"),
        on="order_item_id", how="left"
    ) \
    .join(
        order.select("order_id", "customer_id"),
        on="order_id", how="left"
    ) \
    .join(
        dim_date.select(
            F.col("date_key").alias("return_date_key"),
            F.col("full_date").alias("rd")
        ),
        F.col("date") == F.col("rd"), how="left"
    ) \
    .withColumn("return_key",   F.col("return_id").cast(LongType())) \
    .withColumn("product_key",  F.col("product_id").cast(IntegerType())) \
    .withColumn("customer_key", F.col("customer_id").cast(IntegerType())) \
    .select(
        "return_key",
        F.col("order_item_id").cast(LongType()).alias("order_item_id"),
        "product_key", "customer_key",
        "return_date_key", "status", "refund_amount"
    )
 
fact_returns.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.fact_returns")
 
print(f" fact_returns: {fact_returns.count():,} rows")

 fact_returns: 26,726 rows


In [0]:
#  FACT_EVENTS
# Columns: event_key, customer_key, product_key, date_key,
#          session_id, event_type, page_type, device_type, event_timestamp
 
user_event = spark.table("tijartek_silver.user_event")
session    = spark.table("tijartek_silver.session")
dim_date   = spark.table("tijartek_gold.dim_date")
 
fact_events = user_event \
    .join(
        session.select("session_id", "customer_id", "device_type"),
        on="session_id", how="left"
    ) \
    .join(
        dim_date.select(F.col("date_key"), F.col("full_date").alias("ed")),
        F.col("event_date") == F.col("ed"), how="left"
    ) \
    .withColumn("event_key",    F.col("event_id").cast(LongType())) \
    .withColumn("customer_key", F.col("customer_id").cast(IntegerType())) \
    .withColumn("product_key",  F.col("product_id").cast(IntegerType())) \
    .withColumn("session_id",   F.col("session_id").cast(StringType())) \
    .select(
        "event_key", "customer_key", "product_key", "date_key",
        "session_id", "event_type", "page_type", "device_type", "event_timestamp"
    )
 
fact_events.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("tijartek_gold.fact_events")
 
print(f"fact_events: {fact_events.count():,} rows")
 

fact_events: 500,000 rows


In [0]:
#Verify Gold Layer
print("\n" + "=" * 55)
print("TIJARTEK GOLD LAYER — COMPLETE VERIFICATION")
print("=" * 55)

print("\n   DIMENSION TABLES (6):")
dims = ["dim_date", "dim_location", "dim_customer", "dim_seller",        "dim_promotion", "dim_product"]
for t in dims:
    count = spark.table(f"tijartek_gold.{t}").count()
    print(f"  {t:30} {count:>10,} rows")

print("\n   BRIDGE TABLES (2):")
bridges = ["dim_payment", "dim_review"]
for t in bridges:
    count = spark.table(f"tijartek_gold.{t}").count()
    print(f"  {t:30} {count:>10,} rows")

print("\n   FACT TABLES (4):")
facts = ["fact_sales", "fact_shipments", "fact_returns", "fact_events"]
for t in facts:
    count = spark.table(f"tijartek_gold.{t}").count()
    print(f"  {t:30} {count:>10,} rows")

total = sum(spark.table(f"tijartek_gold.{t}").count() for t in dims + bridges + facts)
print("\n" + "-" * 55)
print(f"  {'TOTAL GOLD TABLES':30} {total:>10,} rows (13 tables)")
print("=" * 55)
print("\n GOLD LAYER COMPLETE — READY FOR POWER BI!")


TIJARTEK GOLD LAYER — COMPLETE VERIFICATION

   DIMENSION TABLES (6):
  dim_date                            2,922 rows
  dim_location                           50 rows
  dim_customer                       80,000 rows
  dim_seller                          1,000 rows
  dim_promotion                         500 rows
  dim_product                        50,000 rows

   BRIDGE TABLES (2):
  dim_payment                       150,000 rows
  dim_review                         60,000 rows

   FACT TABLES (4):
  fact_sales                        338,051 rows
  fact_shipments                    131,981 rows
  fact_returns                       26,726 rows
  fact_events                       500,000 rows

-------------------------------------------------------
  TOTAL GOLD TABLES               1,341,230 rows (13 tables)

 GOLD LAYER COMPLETE — READY FOR POWER BI!
